# 🩺 Apoio à Decisão Clínica — Risco de Síndrome Metabólica
### Protótipo **didático** que vai do *Subjetivo + Objetivo* (S+O) até a *Avaliação* (A) do padrão **SOAP**

---

> ## ⚠️ AVISO IMPORTANTE — LEIA ANTES DE USAR
>
> Este notebook é um **PROTÓTIPO DIDÁTICO** construído com **DADOS 100% SINTÉTICOS** (inventados por código).
> **NÃO é uma ferramenta de diagnóstico** e **NÃO deve ser usado para decisões sobre pacientes reais.**
>
> Em um cenário real, seria **obrigatório**:
> - 🔒 **LGPD (Lei nº 13.709/2018):** dados de saúde são *dados sensíveis*; exigem base legal, consentimento, anonimização e segurança.
> - 🧪 **Validação clínica:** o modelo precisaria ser validado em estudos com **desfechos reais** antes de qualquer uso.
> - 🏛️ **Regulação ANVISA:** um software que apoia decisão clínica pode ser enquadrado como *Software como Dispositivo Médico (SaMD)* e precisa de aprovação regulatória.

**A narrativa:** imagine que **vários profissionais de saúde** registram suas consultas no padrão **SOAP**. Aqui nós *simulamos uma base agregada* dessas consultas e treinamos um modelo simples que aprende a ir do **S**ubjetivo + **O**bjetivo até a **A**valiação (tem ou não risco de síndrome metabólica).

## 🧭 Relembrando o padrão SOAP
- **S — Subjetivo:** o que o paciente *relata* (hábitos, histórico).
- **O — Objetivo:** o que é *medido/examinado* (peso, pressão, exames de sangue...).
- **A — Avaliação:** a *conclusão* do profissional (aqui: risco de síndrome metabólica).
- **P — Plano:** a conduta (não entra neste protótipo).

👉 **Nosso objetivo:** usar **S + O** como *entrada* e prever **A** como *saída*.

### 📋 Roteiro do notebook
0. Importar as bibliotecas
1. Calculadora de gasto energético (TMB e GET)
2. Gerar a base sintética de pacientes (SOAP)
3. Pré-processar (transformar texto em número)
4. Treinar uma Árvore de Decisão
5. Avaliar (acurácia, matriz de confusão, importância das variáveis)
6. Desenhar a árvore
7. Testar um "novo paciente" de forma interativa
8. Conclusão

## Passo 0 — Importando as bibliotecas
Todas estas bibliotecas **já vêm instaladas no Google Colab**. Não precisa instalar nada.

In [ ]:
# numpy: contas e números aleatórios | pandas: tabelas (DataFrame)
import numpy as np
import pandas as pd

# matplotlib: gráficos
import matplotlib.pyplot as plt

# scikit-learn: o "kit" de machine learning (modelo, divisão treino/teste, métricas)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

# Esconde avisos não-críticos só para deixar a saída mais limpa na apresentação.
import warnings
warnings.filterwarnings("ignore")

# "Semente" aleatória: faz os sorteios serem SEMPRE os mesmos.
# Assim o notebook dá o mesmo resultado toda vez que você rodar (reprodutibilidade).
SEMENTE = 42

print("Bibliotecas importadas com sucesso! ✅")

## Passo 1 — Calculadora de Gasto Energético (TMB e GET)

- **TMB (Taxa Metabólica Basal):** energia que o corpo gasta **em repouso** (só para se manter vivo).
- **GET (Gasto Energético Total):** TMB **multiplicada por um fator de atividade física**.

Usamos a fórmula de **Mifflin-St Jeor**, muito comum na prática clínica:

- Homens: `TMB = 10·peso + 6,25·altura − 5·idade + 5`
- Mulheres: `TMB = 10·peso + 6,25·altura − 5·idade − 161`

(peso em kg, altura em cm, idade em anos)

In [ ]:
# Fatores de atividade: multiplicam a TMB. Quanto mais ativo, maior o fator.
FATORES_ATIVIDADE = {
    "sedentario": 1.20,   # pouco ou nenhum exercício
    "leve":       1.375,  # exercício leve 1 a 3x por semana
    "moderado":   1.55,   # exercício moderado 3 a 5x por semana
    "intenso":    1.725,  # exercício intenso 6 a 7x por semana
}

def calcular_gasto_energetico(sexo, peso_kg, altura_cm, idade, nivel_atividade):
    # Calcula a TMB pela fórmula de Mifflin-St Jeor.
    # A fórmula muda só na última constante: +5 (homem) ou -161 (mulher).
    if sexo == "Masculino":
        tmb = 10 * peso_kg + 6.25 * altura_cm - 5 * idade + 5
    else:
        tmb = 10 * peso_kg + 6.25 * altura_cm - 5 * idade - 161

    # GET = TMB vezes o fator de atividade física.
    fator = FATORES_ATIVIDADE[nivel_atividade]
    get = tmb * fator
    return tmb, get

# Exemplo de uso da função:
tmb_ex, get_ex = calcular_gasto_energetico("Masculino", 80, 175, 40, "moderado")
print("Exemplo: homem, 80 kg, 175 cm, 40 anos, atividade moderada")
print(f"  TMB (em repouso): {tmb_ex:.0f} kcal/dia")
print(f"  GET (total/dia) : {get_ex:.0f} kcal/dia")

## Passo 2 — Gerando a base sintética de pacientes (formato SOAP)

Vamos **simular ~800 pacientes** como se viessem de **vários profissionais de saúde**.
Cada paciente tem dados organizados no padrão SOAP:

- **S — Subjetivo:** tabagismo, etilismo, atividade física, horas de sono, histórico familiar de DM/HAS.
- **O — Objetivo:** idade, sexo, peso, altura, IMC, circunferência abdominal, pressão arterial, glicemia, triglicerídeos, HDL e GET.

### 🏷️ Como definimos o rótulo (A — Avaliação)?
Usamos os **critérios clínicos reais de Síndrome Metabólica (IDF / ATP III)**.
O paciente "tem síndrome metabólica" quando atende **3 ou mais** destes 5 critérios:

| Critério | Limite |
|---|---|
| Circunferência abdominal elevada | ≥ 90 cm (homem) / ≥ 80 cm (mulher) |
| Triglicerídeos altos | ≥ 150 mg/dL |
| HDL baixo | < 40 (homem) / < 50 (mulher) |
| Pressão arterial elevada | ≥ 130/85 mmHg |
| Glicemia de jejum elevada | ≥ 100 mg/dL |

> 💡 Adicionamos um **ruído aleatório (~7%)** invertendo alguns rótulos. Isso evita que o problema vire um "decoreba" trivial e deixa o desafio mais realista (na vida real os dados têm imperfeições).

In [ ]:
NUM_PACIENTES = 800

# 'gerador' é o sorteador de números aleatórios (preso à SEMENTE = reprodutível).
gerador = np.random.default_rng(SEMENTE)

# ----------------------- S — SUBJETIVO (o que o paciente relata) -----------------------
sexo          = gerador.choice(["Masculino", "Feminino"], NUM_PACIENTES, p=[0.5, 0.5])
idade         = np.clip(gerador.normal(45, 13, NUM_PACIENTES), 20, 80).round(0)
tabagismo     = gerador.binomial(1, 0.20, NUM_PACIENTES)   # 0 = não, 1 = sim
etilismo      = gerador.binomial(1, 0.30, NUM_PACIENTES)   # 0 = não, 1 = sim
atividade     = gerador.choice(["sedentario", "leve", "moderado", "intenso"],
                               NUM_PACIENTES, p=[0.40, 0.30, 0.20, 0.10])
horas_sono    = np.clip(gerador.normal(6.5, 1.2, NUM_PACIENTES), 3, 10).round(1)
hist_familiar = gerador.binomial(1, 0.35, NUM_PACIENTES)   # histórico de diabetes/hipertensão

# Um "fator de risco oculto": na vida real, exames ruins tendem a aparecer JUNTOS.
# Esse fator (influenciado por idade, sedentarismo, histórico e tabagismo) cria essa
# correlação realista entre os exames. Depois ele é padronizado (média 0, desvio 1).
fator_ativ = np.array([{"sedentario": 1.0, "leve": 0.2, "moderado": -0.4, "intenso": -0.9}[a]
                       for a in atividade])
risco_oculto = (0.025 * (idade - 45) + 0.5 * fator_ativ + 0.4 * hist_familiar
                + 0.25 * tabagismo + gerador.normal(0, 1, NUM_PACIENTES))
risco_oculto = (risco_oculto - risco_oculto.mean()) / risco_oculto.std()

# ----------------------- O — OBJETIVO (o que é medido/examinado) -----------------------
# Altura depende do sexo (média diferente para homens e mulheres).
altura_cm = np.where(sexo == "Masculino",
                     gerador.normal(175, 7, NUM_PACIENTES),
                     gerador.normal(162, 6, NUM_PACIENTES)).round(0)

# IMC e peso (peso = IMC * altura² em metros).
imc  = np.clip(25 + 2.2 * risco_oculto + gerador.normal(0, 2.2, NUM_PACIENTES), 17, 45)
peso = (imc * (altura_cm / 100) ** 2).round(1)

# Circunferência abdominal (ligada ao IMC, ao sexo e ao risco oculto).
circ_abdominal = (np.where(sexo == "Masculino", 88, 78)
                  + 4.0 * risco_oculto + 0.5 * (imc - 25)
                  + gerador.normal(0, 6, NUM_PACIENTES))
circ_abdominal = np.clip(circ_abdominal, 60, 150).round(0)

# Pressão: sistólica e diastólica caminham juntas (compartilham um fator de pressão).
fator_pressao = 0.6 * risco_oculto + gerador.normal(0, 1, NUM_PACIENTES)
pa_sistolica  = np.clip(121 + 8 * fator_pressao + 0.1 * (idade - 45)
                        + gerador.normal(0, 7, NUM_PACIENTES), 90, 200).round(0)
pa_diastolica = np.clip(80 + 5 * fator_pressao + gerador.normal(0, 5, NUM_PACIENTES), 55, 120).round(0)

# Exames de sangue.
glicemia_jejum = np.clip(96 + 6 * risco_oculto + gerador.normal(0, 9, NUM_PACIENTES), 70, 220).round(0)
triglicerideos = np.clip(140 + 22 * risco_oculto + gerador.normal(0, 45, NUM_PACIENTES), 40, 500).round(0)
hdl = (np.where(sexo == "Masculino", 43, 53) - 5 * risco_oculto
       + gerador.normal(0, 9, NUM_PACIENTES))
hdl = np.clip(hdl, 20, 100).round(0)

# GET de cada paciente, usando a calculadora do Passo 1.
get = np.array([calcular_gasto_energetico(s, p, a, i, at)[1]
                for s, p, a, i, at in zip(sexo, peso, altura_cm, idade, atividade)]).round(0)

# ----------------------- A — AVALIAÇÃO (o rótulo que queremos prever) -----------------------
eh_homem = (sexo == "Masculino")
crit_circ = np.where(eh_homem, circ_abdominal >= 90, circ_abdominal >= 80)
crit_tg   = triglicerideos >= 150
crit_hdl  = np.where(eh_homem, hdl < 40, hdl < 50)
crit_pa   = (pa_sistolica >= 130) | (pa_diastolica >= 85)
crit_glic = glicemia_jejum >= 100

# Soma quantos critérios cada paciente atende (0 a 5).
n_criterios = (crit_circ.astype(int) + crit_tg.astype(int) + crit_hdl.astype(int)
               + crit_pa.astype(int) + crit_glic.astype(int))

# Tem síndrome metabólica = atende 3 ou mais critérios.
tem_sindrome = (n_criterios >= 3).astype(int)

# Ruído: inverte ~7% dos rótulos para não virar problema trivial.
inverter = gerador.random(NUM_PACIENTES) < 0.07
tem_sindrome = np.where(inverter, 1 - tem_sindrome, tem_sindrome)

# ----------------------- Monta a tabela (DataFrame) -----------------------
df = pd.DataFrame({
    # S
    "sexo": sexo, "idade": idade, "tabagismo": tabagismo, "etilismo": etilismo,
    "atividade_fisica": atividade, "horas_sono": horas_sono, "hist_familiar": hist_familiar,
    # O
    "peso": peso, "altura_cm": altura_cm, "imc": imc.round(1), "circ_abdominal": circ_abdominal,
    "pa_sistolica": pa_sistolica, "pa_diastolica": pa_diastolica, "glicemia_jejum": glicemia_jejum,
    "triglicerideos": triglicerideos, "hdl": hdl, "get": get,
    # A (rótulo)
    "tem_sindrome_metabolica": tem_sindrome,
})

print(f"Base criada com {df.shape[0]} pacientes e {df.shape[1]} colunas.")
print(f"Proporção de casos POSITIVOS (com síndrome): {df['tem_sindrome_metabolica'].mean():.1%}")
print("\nPrimeiras linhas da base:")
df.head()

## Passo 3 — Pré-processamento (transformar texto em número)

Modelos de machine learning entendem **números**, não texto. Então convertemos as colunas
de texto (`sexo` e `atividade_fisica`) em números e separamos:
- **X** = as *entradas* (tudo que o profissional observa: S + O)
- **y** = a *resposta* que queremos prever (A: tem ou não síndrome metabólica)

In [ ]:
# 'sexo' é uma categoria simples -> vira 0 e 1.
mapa_sexo = {"Feminino": 0, "Masculino": 1}

# 'atividade_fisica' tem ORDEM natural (do menos para o mais ativo),
# então mapeamos em ordem crescente (isso se chama codificação ordinal).
mapa_atividade = {"sedentario": 0, "leve": 1, "moderado": 2, "intenso": 3}

dados = df.copy()
dados["sexo"] = dados["sexo"].map(mapa_sexo)
dados["atividade_fisica"] = dados["atividade_fisica"].map(mapa_atividade)

# X recebe todas as colunas MENOS o rótulo. y recebe só o rótulo.
X = dados.drop(columns=["tem_sindrome_metabolica"])
y = dados["tem_sindrome_metabolica"]

print("Formato de X (linhas, colunas):", X.shape)
print("Colunas usadas como entrada (S + O):")
print(list(X.columns))

## Passo 4 — Treinando uma Árvore de Decisão

- Separamos **70% para treinar** e **30% para testar**. O teste simula *pacientes novos*, que o modelo nunca viu.
- Usamos uma **Árvore de Decisão**: ela faz uma sequência de perguntas do tipo *"exame > valor?"*.
- `max_depth=4` deixa a árvore **pequena e legível** (boa para explicar).
- `min_samples_leaf=20` exige **pelo menos 20 pacientes em cada "folha"**. Isso impede a árvore de criar regrinhas baseadas em pouquíssimos casos (ou seja, evita "decorar" o ruído).

> 🌳 **Curiosidade:** Árvore de Decisão **NÃO precisa de normalização**. Ela compara cada variável com um limite, então a *escala* dos números (kg, mmHg, mg/dL...) não atrapalha.

In [ ]:
# stratify=y mantém a mesma proporção de positivos no treino e no teste.
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.30, random_state=SEMENTE, stratify=y
)

# Cria e treina o modelo.
# min_samples_leaf=20 deixa o modelo mais "calmo": cada decisão final apoia-se em
# pelo menos 20 pacientes, o que melhora a generalização e evita reagir a ruído.
modelo = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=SEMENTE)
modelo.fit(X_treino, y_treino)

print(f"Modelo treinado com {len(X_treino)} pacientes.")
print(f"Guardamos {len(X_teste)} pacientes para o teste.")

## Passo 5 — Avaliando o modelo
Vamos olhar três coisas: **acurácia**, **matriz de confusão** e **importância das variáveis**.

In [ ]:
# (1) ACURÁCIA: porcentagem de acertos nos pacientes de teste.
y_previsto = modelo.predict(X_teste)
acuracia = accuracy_score(y_teste, y_previsto)
print(f"Acurácia no teste: {acuracia:.1%}")

# (2) MATRIZ DE CONFUSÃO: mostra acertos e erros separados por classe.
ConfusionMatrixDisplay.from_predictions(
    y_teste, y_previsto, display_labels=["Sem risco", "Com risco"], cmap="Blues"
)
plt.title("Matriz de Confusão (teste)")
plt.show()

In [ ]:
# (3) IMPORTÂNCIA DAS VARIÁVEIS: o quanto cada variável pesou nas decisões do modelo.
importancias = pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=False)

print("Importância de cada variável (maior = mais usada pelo modelo):")
print(importancias.round(3))

# Gráfico de barras horizontais (do menor para o maior, para ler de baixo para cima).
importancias.sort_values().plot(kind="barh", figsize=(8, 6), color="teal")
plt.title("Importância das variáveis")
plt.xlabel("Importância")
plt.tight_layout()
plt.show()

> 🔎 **Repare no resultado!** As variáveis no topo são justamente os **5 critérios clínicos**: glicemia, pressão, triglicerídeos, HDL e circunferência abdominal.
>
> O modelo **"redescobriu" sozinho** as regras que os médicos usam — **sem que ninguém tenha contado** quais eram. Já variáveis como *peso*, *altura* ou *GET* ficaram com importância quase **zero**, porque o rótulo foi definido pelos exames, e não por elas.

## Passo 6 — Desenhando a árvore de decisão

Cada **caixa** é uma pergunta sobre um exame. Seguindo os caminhos (esquerda = "sim", direita = "não"),
chegamos a uma decisão: **Sem risco** ou **Com risco**. A cor indica a classe mais provável naquele ponto.

In [ ]:
plt.figure(figsize=(22, 11))
plot_tree(
    modelo,
    feature_names=X.columns,
    class_names=["Sem risco", "Com risco"],
    filled=True,     # pinta as caixas pela classe
    rounded=True,
    fontsize=9,
)
plt.title("Árvore de Decisão treinada (cada caixa é uma pergunta sobre um exame)")
plt.show()

## Passo 7 — Testando um "novo paciente"

Agora a parte divertida: montar um paciente e ver o que o modelo prevê.
Primeiro definimos duas funções auxiliares; depois vêm os **controles interativos**.

In [ ]:
def avaliar_paciente(paciente):
    # Recebe um dicionário com os dados do paciente e devolve:
    # GET, previsão (0/1), probabilidade de risco e quais critérios clínicos ele atende.

    # 1) Calcula o GET com a calculadora do Passo 1.
    _, get_paciente = calcular_gasto_energetico(
        paciente["sexo"], paciente["peso"], paciente["altura_cm"],
        paciente["idade"], paciente["atividade_fisica"]
    )

    # 2) Monta UMA linha no MESMO formato/ordem que o modelo aprendeu.
    linha = {
        "sexo": mapa_sexo[paciente["sexo"]],
        "idade": paciente["idade"],
        "tabagismo": paciente["tabagismo"],
        "etilismo": paciente["etilismo"],
        "atividade_fisica": mapa_atividade[paciente["atividade_fisica"]],
        "horas_sono": paciente["horas_sono"],
        "hist_familiar": paciente["hist_familiar"],
        "peso": paciente["peso"],
        "altura_cm": paciente["altura_cm"],
        "imc": round(paciente["peso"] / (paciente["altura_cm"] / 100) ** 2, 1),
        "circ_abdominal": paciente["circ_abdominal"],
        "pa_sistolica": paciente["pa_sistolica"],
        "pa_diastolica": paciente["pa_diastolica"],
        "glicemia_jejum": paciente["glicemia_jejum"],
        "triglicerideos": paciente["triglicerideos"],
        "hdl": paciente["hdl"],
        "get": round(get_paciente, 0),
    }
    X_novo = pd.DataFrame([linha])[X.columns]   # garante a mesma ordem de colunas

    # 3) Pergunta ao modelo: classe prevista e probabilidade de "Com risco".
    predicao = modelo.predict(X_novo)[0]
    probabilidade = modelo.predict_proba(X_novo)[0][1]

    # 4) Confere quais critérios clínicos o paciente atende (independente do modelo).
    eh_h = (paciente["sexo"] == "Masculino")
    criterios = {
        "Circunferência abdominal elevada": paciente["circ_abdominal"] >= (90 if eh_h else 80),
        "Triglicerídeos ≥ 150":             paciente["triglicerideos"] >= 150,
        "HDL baixo":                        paciente["hdl"] < (40 if eh_h else 50),
        "Pressão ≥ 130/85":                 (paciente["pa_sistolica"] >= 130) or (paciente["pa_diastolica"] >= 85),
        "Glicemia ≥ 100":                   paciente["glicemia_jejum"] >= 100,
    }
    return get_paciente, predicao, probabilidade, criterios


def mostrar_resultado(paciente):
    # Chama a função acima e imprime tudo de forma organizada.
    get_paciente, predicao, probabilidade, criterios = avaliar_paciente(paciente)

    print("=" * 52)
    print(f"GET estimado: {get_paciente:.0f} kcal/dia")
    print("-" * 52)
    if predicao == 1:
        print("RESULTADO DO MODELO: ⚠️ RISCO ELEVADO de síndrome metabólica")
    else:
        print("RESULTADO DO MODELO: ✅ baixo risco")
    print(f"Probabilidade estimada de risco: {probabilidade:.0%}")
    print("-" * 52)
    print("Critérios clínicos atendidos por este paciente:")
    total = 0
    for nome, atende in criterios.items():
        marca = "✅" if atende else "⬜"
        print(f"   {marca}  {nome}")
        if atende:
            total += 1
    print(f"\nTotal de critérios atendidos: {total} de 5")
    print("(3 ou mais = síndrome metabólica pelos critérios clínicos)")
    print("=" * 52)

print("Funções prontas! Rode a próxima célula para usar os controles interativos.")

### 🎛️ Versão interativa (ipywidgets)
Mova os controles para montar o paciente e clique em **Avaliar paciente**.

In [ ]:
# Os controles interativos usam a biblioteca ipywidgets (já vem no Colab).
# Se por algum motivo não funcionar no seu ambiente, use a célula seguinte (versão sem widgets).
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    # --- S (Subjetivo) ---
    w_sexo  = widgets.Dropdown(options=["Masculino", "Feminino"], value="Masculino", description="Sexo:")
    w_idade = widgets.IntSlider(value=52, min=20, max=80, description="Idade:")
    w_ativ  = widgets.Dropdown(options=["sedentario", "leve", "moderado", "intenso"],
                               value="sedentario", description="Atividade:")
    w_tabag = widgets.Dropdown(options=[("Não", 0), ("Sim", 1)], value=1, description="Tabagismo:")
    w_etil  = widgets.Dropdown(options=[("Não", 0), ("Sim", 1)], value=0, description="Etilismo:")
    w_hist  = widgets.Dropdown(options=[("Não", 0), ("Sim", 1)], value=1, description="Hist. fam.:")
    w_sono  = widgets.FloatSlider(value=6.0, min=3, max=10, step=0.5, description="Horas sono:")
    # --- O (Objetivo) ---
    w_peso  = widgets.FloatSlider(value=95, min=40, max=160, step=0.5, description="Peso (kg):")
    w_alt   = widgets.IntSlider(value=175, min=140, max=210, description="Altura (cm):")
    w_circ  = widgets.IntSlider(value=104, min=60, max=150, description="Circ. abd:")
    w_pas   = widgets.IntSlider(value=138, min=90, max=200, description="PA sist:")
    w_pad   = widgets.IntSlider(value=88, min=55, max=120, description="PA diast:")
    w_glic  = widgets.IntSlider(value=110, min=70, max=220, description="Glicemia:")
    w_tg    = widgets.IntSlider(value=180, min=40, max=500, description="Triglic:")
    w_hdl   = widgets.IntSlider(value=38, min=20, max=100, description="HDL:")

    botao = widgets.Button(description="Avaliar paciente", button_style="success")
    saida = widgets.Output()

    def ao_clicar(_):
        with saida:
            clear_output()
            paciente = {
                "sexo": w_sexo.value, "idade": w_idade.value, "tabagismo": w_tabag.value,
                "etilismo": w_etil.value, "atividade_fisica": w_ativ.value,
                "horas_sono": w_sono.value, "hist_familiar": w_hist.value,
                "peso": w_peso.value, "altura_cm": w_alt.value, "circ_abdominal": w_circ.value,
                "pa_sistolica": w_pas.value, "pa_diastolica": w_pad.value,
                "glicemia_jejum": w_glic.value, "triglicerideos": w_tg.value, "hdl": w_hdl.value,
            }
            mostrar_resultado(paciente)

    botao.on_click(ao_clicar)

    coluna_s = widgets.VBox([widgets.HTML("<b>S — Subjetivo</b>"),
                             w_sexo, w_idade, w_ativ, w_tabag, w_etil, w_hist, w_sono])
    coluna_o = widgets.VBox([widgets.HTML("<b>O — Objetivo</b>"),
                             w_peso, w_alt, w_circ, w_pas, w_pad, w_glic, w_tg, w_hdl])
    display(widgets.VBox([widgets.HBox([coluna_s, coluna_o]), botao, saida]))

except Exception as erro:
    print("Não foi possível carregar os controles interativos:", erro)
    print("Sem problema! Use a próxima célula (versão sem widgets).")

### ⌨️ Versão alternativa (sem widgets)
Se os controles acima não aparecerem, **edite o dicionário abaixo** e rode a célula.
(Dica: também daria para pedir os valores com `input()`, mas o dicionário é mais prático para apresentar.)

In [ ]:
# Edite os valores deste paciente e rode a célula para ver o resultado.
novo_paciente = {
    # S — Subjetivo
    "sexo": "Masculino",          # "Masculino" ou "Feminino"
    "idade": 52,
    "tabagismo": 1,               # 0 = não, 1 = sim
    "etilismo": 0,                # 0 = não, 1 = sim
    "atividade_fisica": "sedentario",  # sedentario / leve / moderado / intenso
    "horas_sono": 6.0,
    "hist_familiar": 1,           # 0 = não, 1 = sim
    # O — Objetivo
    "peso": 95.0,
    "altura_cm": 175,
    "circ_abdominal": 104,
    "pa_sistolica": 138,
    "pa_diastolica": 88,
    "glicemia_jejum": 110,
    "triglicerideos": 180,
    "hdl": 38,
}

mostrar_resultado(novo_paciente)

## ✅ Conclusão (resumo para o slide)

- 🧠 **O que o modelo faz:** a partir do **S + O** de uma consulta (padrão SOAP), ele estima o **risco de síndrome metabólica** — e, ao olhar a *importância das variáveis*, vemos que ele **redescobriu sozinho os 5 critérios clínicos**.
- 🧩 **Por que o algoritmo é a parte "fácil":** treinar a árvore foram poucas linhas de código. O **difícil e valioso** é tudo em volta: dados de qualidade, definição do desfecho, validação clínica e cuidado ético/legal.
- ⚠️ **Limites reais (não ignorar):**
  - **Desfecho:** aqui o rótulo é uma *regra conhecida*; na vida real o desfecho clínico é incerto e demora a aparecer.
  - **Viés e volume:** uma base pequena ou enviesada gera um modelo injusto/errado. "Vários profissionais" ajuda no volume, mas exige **padronização** dos registros.
  - **Regulação e privacidade:** **LGPD** (dados sensíveis) e **ANVISA** (software como dispositivo médico) tornam um uso real bem mais complexo que o protótipo.
- 🎯 **Mensagem final:** este é um **apoio à decisão**, **nunca um substituto** do julgamento do profissional de saúde.